# Data Profiling
Loads the table and displays schema, sample data, and per-column summary (nulls, distinct counts).

**Configure the cell below** with your table and lakehouse names, then **Run All**.

In [ ]:
# ── Configuration ──────────────────────────────────────────────
TABLE_NAME = "{{TABLE_NAME}}"
LAKEHOUSE_NAME = "{{LAKEHOUSE_NAME}}"
# ───────────────────────────────────────────────────────────────

from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql.types import *

spark = SparkSession.builder.getOrCreate()
df = spark.table(TABLE_NAME)
original_count = df.count()

print(f"Table: {TABLE_NAME}")
print(f"Rows: {original_count:,}")
print(f"Columns: {len(df.columns)}")

In [ ]:
print("=" * 60)
print("TABLE SCHEMA")
print("=" * 60)
df.printSchema()

print("\n" + "=" * 60)
print("SAMPLE DATA (first 5 rows)")
print("=" * 60)
display(df.limit(5))

In [ ]:
print("=" * 60)
print("COLUMN SUMMARY")
print("=" * 60)

col_summary = []
for field in df.schema.fields:
    null_count = df.where(F.col(field.name).isNull()).count()
    distinct_count = df.select(field.name).distinct().count()
    col_summary.append({
        "column": field.name,
        "type": str(field.dataType),
        "nullable": field.nullable,
        "null_count": null_count,
        "null_pct": round(null_count / original_count * 100, 2) if original_count > 0 else 0,
        "distinct_count": distinct_count
    })

summary_df = spark.createDataFrame(col_summary)
display(summary_df)